In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from time import perf_counter

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI, OpenAIError
from pydantic import BaseModel, Field
from tqdm.auto import tqdm


load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY was not found. "
        "Make sure the project-root .env file is present."
    )


CHUNKS_PATH = Path("../data/chunks.parquet")
EVALUATION_DIR = Path("../data/evaluation")

SAMPLE_PATH = EVALUATION_DIR / "evaluation-sample.parquet"
CHECKPOINT_PATH = (
    EVALUATION_DIR / "ground-truth-checkpoint.jsonl"
)
GROUND_TRUTH_PATH = (
    EVALUATION_DIR / "ground-truth.json"
)

MODEL = "gpt-4o-mini"
SAMPLE_SIZE = 260
TARGET_GROUND_TRUTH_SIZE = 181
RANDOM_STATE = 42

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

client = OpenAI()

print("Configuration loaded.")

Configuration loaded.


In [44]:
chunks = pd.read_parquet(CHUNKS_PATH)

print("Total chunks:", len(chunks))
print("Columns:", chunks.columns.tolist())

chunks.head(3)

Total chunks: 50910
Columns: ['chunk_id', 'episode_id', 'chunk_index', 'chunk_position', 'guest', 'episode_title', 'publish_date', 'youtube_url', 'video_id', 'speaker_name', 'start_time', 'end_time', 'text', 'word_count', 'is_sponsor_read']


,chunk_id,episode_id,chunk_index,chunk_position,guest,episode_title,publish_date,youtube_url,video_id,speaker_name,start_time,end_time,text,word_count,is_sponsor_read
0,0685d77dd0a1220827c2c360117670d8,9b37badc1aba31023d118db15a3fa95e,0,0,Ada Chen Rekhi,Feeling stuck? Here's how to know when it's ti...,2023-04-21,https://www.youtube.com/watch?v=l-T8sNRcWQk,l-T8sNRcWQk,Ada Chen Rekhi,0.0,36.0,It's a terrible outcome to wake up one day and...,105,False
1,2044874fa10d0def0a2848a7de4c3d58,9b37badc1aba31023d118db15a3fa95e,1,1,Ada Chen Rekhi,Feeling stuck? Here's how to know when it's ti...,2023-04-21,https://www.youtube.com/watch?v=l-T8sNRcWQk,l-T8sNRcWQk,Lenny,36.0,81.0,"Welcome to Lenny's Podcast, where I interview ...",149,False
2,54adba3dfa8d3c6c8874208e2d494cfe,9b37badc1aba31023d118db15a3fa95e,2,2,Ada Chen Rekhi,Feeling stuck? Here's how to know when it's ti...,2023-04-21,https://www.youtube.com/watch?v=l-T8sNRcWQk,l-T8sNRcWQk,Lenny,81.0,100.0,We do a live exercise around my own personal v...,69,True


In [45]:
def normalize_name(value: object) -> str:
    if value is None or pd.isna(value):
        return ""

    return str(value).strip().casefold()


candidates = chunks.copy()

# Remove sponsor and promotional chunks.
candidates = candidates[
    ~candidates["is_sponsor_read"].fillna(False)
].copy()

# Keep meaningful excerpts.
candidates = candidates[
    candidates["text"].fillna("").str.strip().ne("")
].copy()

candidates = candidates[
    candidates["word_count"].fillna(0).between(50, 300)
].copy()

# Prefer advice stated directly by the episode guest.
guest_names = candidates["guest"].map(normalize_name)
speaker_names = candidates["speaker_name"].map(
    normalize_name
)

candidates = candidates[
    guest_names.eq(speaker_names)
].copy()

# Remove accidental duplicate chunk IDs defensively.
candidates = candidates.drop_duplicates(
    subset=["chunk_id"]
).copy()

print("Eligible evaluation chunks:", len(candidates))
print("Eligible episodes:", candidates["episode_id"].nunique())

Eligible evaluation chunks: 20464
Eligible episodes: 233


In [46]:
candidates["length_bucket"] = pd.cut(
    candidates["word_count"],
    bins=[49, 100, 200, 300],
    labels=["short", "medium", "long"],
    include_lowest=True,
)

print(
    candidates["length_bucket"]
    .value_counts()
    .sort_index()
)

length_bucket
short      7456
medium    12607
long        401
Name: count, dtype: int64


In [47]:
def sample_bucket_with_episode_diversity(
    bucket_rows: pd.DataFrame,
    target_count: int,
    random_state: int,
) -> pd.DataFrame:
    """
    Sample one length bucket while maximizing episode diversity.

    First select at most one chunk per episode. If that does not produce
    enough rows, fill the remaining slots from other chunks in the same
    length bucket.
    """
    shuffled = bucket_rows.sample(
        frac=1,
        random_state=random_state,
    ).copy()

    one_per_episode = shuffled.drop_duplicates(
        subset=["episode_id"],
        keep="first",
    )

    initial_count = min(
        target_count,
        len(one_per_episode),
    )

    selected = one_per_episode.head(
        initial_count
    ).copy()

    remaining_slots = (
        target_count - len(selected)
    )

    if remaining_slots > 0:
        remaining_rows = shuffled[
            ~shuffled["chunk_id"].isin(
                selected["chunk_id"]
            )
        ]

        fill_count = min(
            remaining_slots,
            len(remaining_rows),
        )

        selected = pd.concat(
            [
                selected,
                remaining_rows.head(fill_count),
            ],
            ignore_index=True,
        )

    return selected


def create_stratified_sample(
    dataframe: pd.DataFrame,
    sample_size: int,
    random_state: int,
) -> pd.DataFrame:
    """
    Create a reproducible sample balanced across chunk-length buckets.

    The function aims for equal numbers of short, medium, and long
    chunks while maximizing episode diversity within each bucket.
    """
    buckets = [
        "short",
        "medium",
        "long",
    ]

    target_per_bucket = (
        sample_size // len(buckets)
    )

    sampled_parts: list[pd.DataFrame] = []

    for bucket_number, bucket in enumerate(buckets):
        bucket_rows = dataframe[
            dataframe["length_bucket"].eq(bucket)
        ].copy()

        print(
            f"{bucket}: "
            f"{len(bucket_rows)} eligible chunks from "
            f"{bucket_rows['episode_id'].nunique()} episodes"
        )

        sampled_bucket = (
            sample_bucket_with_episode_diversity(
                bucket_rows=bucket_rows,
                target_count=target_per_bucket,
                random_state=(
                    random_state + bucket_number
                ),
            )
        )

        sampled_parts.append(
            sampled_bucket
        )

    sampled = pd.concat(
        sampled_parts,
        ignore_index=True,
    )

    # Handle sample sizes not evenly divisible by three.
    remaining_slots = (
        sample_size - len(sampled)
    )

    if remaining_slots > 0:
        remaining_rows = dataframe[
            ~dataframe["chunk_id"].isin(
                sampled["chunk_id"]
            )
        ].sample(
            frac=1,
            random_state=random_state + 100,
        )

        sampled = pd.concat(
            [
                sampled,
                remaining_rows.head(
                    remaining_slots
                ),
            ],
            ignore_index=True,
        )

    sampled = (
        sampled
        .drop_duplicates(subset=["chunk_id"])
        .sample(
            frac=1,
            random_state=random_state,
        )
        .reset_index(drop=True)
    )

    return sampled


evaluation_sample = create_stratified_sample(
    dataframe=candidates,
    sample_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)

print()
print("Sampled chunks:", len(evaluation_sample))

print(
    evaluation_sample["length_bucket"]
    .value_counts()
    .sort_index()
)

print(
    "Unique episodes:",
    evaluation_sample["episode_id"].nunique(),
)

print(
    "Duplicate chunk IDs:",
    evaluation_sample["chunk_id"].duplicated().sum(),
)

short: 7456 eligible chunks from 230 episodes
medium: 12607 eligible chunks from 230 episodes
long: 401 eligible chunks from 122 episodes

Sampled chunks: 260
length_bucket
short     87
medium    87
long      86
Name: count, dtype: int64
Unique episodes: 174
Duplicate chunk IDs: 0


In [48]:
print(
    evaluation_sample.groupby(
        "length_bucket",
        observed=True,
    ).agg(
        chunks=("chunk_id", "count"),
        unique_episodes=("episode_id", "nunique"),
        minimum_words=("word_count", "min"),
        maximum_words=("word_count", "max"),
        average_words=("word_count", "mean"),
    )
)

               chunks  unique_episodes  minimum_words  maximum_words  \
length_bucket                                                          
short              87               87             50            100   
medium             87               87            101            195   
long               86               86            201            296   

               average_words  
length_bucket                 
short              79.137931  
medium            137.218391  
long              228.930233  


In [49]:
display_columns = [
    "chunk_id",
    "guest",
    "episode_title",
    "speaker_name",
    "word_count",
    "length_bucket",
    "text",
]

evaluation_sample[
    display_columns
].head(10)

,chunk_id,guest,episode_title,speaker_name,word_count,length_bucket,text
0,248716fafa12abd5c713bd05bf772ce9,Dan Shipper,"The AI-native startup: 5 products, 7-figure re...",Dan Shipper,66,short,And I think for us it's a little easier becaus...
1,b6619b3c795e8067df765b4de16e3e13,Jason Feifer,How to get press for your product | Jason Feif...,Jason Feifer,209,long,"I got to Entrepreneur Magazine, zero, zero peo..."
2,e6e7f0d1e36a399dc842a84f8b7616ae,Kristen Berman,Using behavioral science to improve your produ...,Kristen Berman,222,long,"So, big experiment. This is with 10,000 people..."
3,44b88aff3e09db750800f76a50370b70,Josh Miller,Competing with giants: An inside look at how T...,Josh Miller,208,long,A lot of people tend to. I've noticed a lot of...
4,63040247210c6876652512e28be609fd,Drew Houston,Behind the founder | Drew Houston (Dropbox),Drew Houston,270,long,"Initially, I thought I was like, all right, we..."
5,6b9d45b5826db416ce56680e0c0e660c,Chip Conley,Mastering product strategy and growing as a PM...,Chip Conley,99,short,When it's successful. It's brilliant. I think ...
6,230318fd1b8c9bf735bb6a417de68089,Hila Qu,The ultimate guide to adding a PLG motion | Hi...,Hila Qu,294,long,... where is confusing. Where do you get stuck...
7,2476625dff494c768dd07c86b29a8bed,Bob Baxley,"35 years of product design wisdom from Apple, ...",Bob Baxley,161,medium,We could talk about this when you take a final...
8,1f1d86cf5d616477987a6680543f9320,Yuriy Timen,How to grow a subscription business | Yuriy Ti...,Yuriy Timen,97,short,Then it's really powerful. I think that there ...
9,904b5f82500e256d66142afe224ba542,Karri Saarinen,"Inside Linear: Building with taste, craft, and...",Karri Saarinen,239,long,But then we have projects where we are not sur...


In [50]:
if CHECKPOINT_PATH.exists():
    backup_path = CHECKPOINT_PATH.with_name(
        "ground-truth-checkpoint-before-relevance.jsonl"
    )
    CHECKPOINT_PATH.rename(backup_path)
    print("Old checkpoint moved to:", backup_path)
else:
    print("No previous checkpoint found.")

Old checkpoint moved to: ../data/evaluation/ground-truth-checkpoint-before-relevance.jsonl


In [33]:
if GROUND_TRUTH_PATH.exists():
    GROUND_TRUTH_PATH.unlink()
    print("Removed old ground-truth file.")

Removed old ground-truth file.


In [8]:
evaluation_sample[
    display_columns
].sort_values("word_count").head(10)

,chunk_id,guest,episode_title,speaker_name,word_count,length_bucket,text
14,b29b4004cf4f4acbbf8107d13ae31097,Sarah Tavel,The hierarchy of engagement | Sarah Tavel (Ben...,Sarah Tavel,50,short,"What I often feel is that, a lot of times, pro..."
47,36fb18ffed100e175bb2c950fec36abb,Oji Udezue,"Picking sharp problems, increasing virality, a...",Oji Udezue,50,short,"It was fun, but I don't think it was a sharp p..."
119,53219d9cc71e62a07cf1e1a2d8a4cdf3,Noah Weiss,"The 10 traits of great PMs, AI, and Slack’s ap...",Noah Weiss,51,short,A classic example of that would be Netflix bac...
141,2dab41625d0f6c144325ad7381cf1d4e,Ken Norton,How to unlock your product leadership skills |...,Ken Norton,51,short,It does feel like a little bit ... And I felt ...
39,1fa2fab1554cea6e79a59fd20029cf62,Uri Levine,A founder’s guide to crisis management | Uri L...,Uri Levine,53,short,"Raising capital, it's a journey by itself and ..."
120,66ac009acfda1019091f1c936f912c21,Daniel Lereya,Daniel Lereya,Daniel Lereya,55,short,Some of our competitors did something that we ...
155,f6674e84f89478da0b4a87c60660eedb,Jeff Weinstein,"Building product at Stripe: craft, metrics, an...",Jeff Weinstein,55,short,"I like counter-positioning also, and Atlas wit..."
122,f2c37cb6948f81fba7e9ffd8a8dc696d,Sean Ellis,The original growth hacker reveals his secrets...,Sean Ellis,55,short,Just ignore the people who say they'd be somew...
41,c3bfc7bc0e8ca06eb0a8159aadc0af0c,Lane Shackleton,What sets great teams apart | Lane Shackleton ...,Lane Shackleton,56,short,I mean definitely sports. I would say sports i...
5,e4e3992599558bd14d013f3a60210a99,Archie Abrams,How to speak more confidently and persuasively...,Archie Abrams,57,short,We have two big groups within growth. So one i...


In [52]:
evaluation_sample.to_parquet(
    SAMPLE_PATH,
    index=False,
)

print("Saved evaluation sample to:", SAMPLE_PATH)

Saved evaluation sample to: ../data/evaluation/evaluation-sample.parquet


In [60]:
checkpoint_records = load_checkpoint_records(
    CHECKPOINT_PATH
)

checkpoint_df = pd.DataFrame(checkpoint_records)

print("Checkpoint records:", len(checkpoint_df))

if "is_relevant" in checkpoint_df.columns:
    print(
        checkpoint_df["is_relevant"]
        .value_counts(dropna=False)
    )

NameError: name 'load_checkpoint_records' is not defined

In [54]:
class GeneratedQuestion(BaseModel):
    is_relevant: bool = Field(
        description=(
            "Whether the excerpt can produce a useful evaluation "
            "question about product management, growth, strategy, "
            "leadership, customer research, business building, "
            "design, or team operations."
        )
    )

    question: str = Field(
        description=(
            "A standalone product-management question that "
            "the supplied excerpt directly answers."
        )
    )

    expected_answer: str = Field(
        description=(
            "A concise expected answer supported only by "
            "the supplied excerpt."
        )
    )

    topic: str = Field(
        description=(
            "A short topic label such as pricing, retention, "
            "leadership, user research, or strategy."
        )
    )

In [55]:
GENERATION_SYSTEM_PROMPT = """
You create evaluation questions for a product-management retrieval system.

Given one transcript excerpt from Lenny's Podcast, generate exactly one
specific, natural-language question that the excerpt directly answers.

Requirements:
- The question must be understandable without seeing the excerpt.
- The question must test the meaning of the excerpt, not merely copy a
  distinctive phrase from it.
- Do not mention the chunk ID.
- Do not mention the episode title.
- Do not mention the guest's name unless the advice only makes sense as
  a guest-specific question.
- Avoid vague questions such as "What did they discuss?"
- Avoid yes/no questions.
- Do not use information outside the supplied excerpt.
- Write the expected answer in clear, natural language.
- Include only claims explicitly supported by the excerpt.
- Do not repeat unclear, accidental, or context-dependent phrases from
  the transcript unless they are essential to the answer.
- Avoid questions that depend heavily on personal biographical context
  unless the excerpt clearly explains that context.
- The expected answer must be concise and fully supported by the excerpt.
- Set is_relevant to true only when the excerpt supports a useful
  question about product management, growth, product design, strategy,
  leadership, customer research, business building, or team operations.
- Set is_relevant to false for medical details, unrelated historical
  facts, personal anecdotes without a transferable professional lesson,
  advertisements, introductions, and conversational filler.
- When is_relevant is false, return an empty question and expected answer
  and use "irrelevant" as the topic.
- Do not add abstract conclusions, causes, or lessons that are not stated
  or clearly implied by the excerpt.
""".strip()

def build_generation_prompt(
    row: pd.Series,
) -> str:
    return (
        f"Guest: {row['guest']}\n"
        f"Episode: {row['episode_title']}\n"
        f"Speaker: {row['speaker_name']}\n\n"
        "Transcript excerpt:\n"
        f"{row['text']}"
    )

In [56]:
def generate_question(
    row: pd.Series,
) -> tuple[GeneratedQuestion, dict[str, object]]:
    """
    Generate one ground-truth question and return usage metadata.
    """
    started_at = perf_counter()

    completion = client.chat.completions.parse(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": GENERATION_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": build_generation_prompt(row),
            },
        ],
        response_format=GeneratedQuestion,
    )

    message = completion.choices[0].message

    if message.refusal:
        raise RuntimeError(
            f"Model refused generation: {message.refusal}"
        )

    generated = message.parsed

    if generated is None:
        raise RuntimeError(
            "The model returned no parsed question."
        )

    elapsed_ms = round(
        (perf_counter() - started_at) * 1000
    )

    usage = completion.usage

    metadata = {
        "model": MODEL,
        "input_tokens": (
            usage.prompt_tokens
            if usage is not None
            else 0
        ),
        "output_tokens": (
            usage.completion_tokens
            if usage is not None
            else 0
        ),
        "total_tokens": (
            usage.total_tokens
            if usage is not None
            else 0
        ),
        "latency_ms": elapsed_ms,
        "response_id": completion.id,
    }

    return generated, metadata

In [37]:
test_rows = evaluation_sample.sample(
    n=5,
    random_state=42,
)

for _, test_row in test_rows.iterrows():
    generated, metadata = generate_question(test_row)

    print("=" * 100)
    print("GUEST:", test_row["guest"])
    print("RELEVANT:", generated.is_relevant)
    print("QUESTION:", generated.question)
    print("EXPECTED ANSWER:", generated.expected_answer)
    print("TOPIC:", generated.topic)
    print("SOURCE:", test_row["text"][:700])
    print("METADATA:", metadata)
    print()

GUEST: Alex Komoroske
RELEVANT: True
QUESTION: What strategy does Alex Komoroske suggest for making decisions in uncertain environments?
EXPECTED ANSWER: Alex Komoroske suggests slicing decisions into smaller, manageable steps to reduce risk and allow for adjusting directions based on outcomes.
TOPIC: decision-making
SOURCE: And so if you slice this thing up and you have a coherent worldview and you have a principled approach, you can arc to wildly different outcomes than look like they were possible while at each point, each individual action is safe and reasonable. And so you can combine both of these things. I think so many times we try to jump and we jump to the end state of the thing. And actually you don't need to make that decision. If you can slice up your decisions into smaller and smaller decisions, I'm like, "This next step definitely makes sense." It will almost certainly pay for itself or the very least won't be too expensive. And then it might allow these other things to 

In [15]:
test_row = evaluation_sample.iloc[0]

generated, metadata = generate_question(
    test_row
)

print("QUESTION:")
print(generated.question)

print("\nEXPECTED ANSWER:")
print(generated.expected_answer)

print("\nTOPIC:")
print(generated.topic)

print("\nMETADATA:")
print(metadata)

QUESTION:
What realization did Deb Liu come to about her job while managing work and motherhood?

EXPECTED ANSWER:
Deb Liu realized that she was bored of her job and felt uncertain about its direction, especially while managing her responsibilities as a parent.

TOPIC:
career management

METADATA:
{'model': 'gpt-4o-mini', 'input_tokens': 456, 'output_tokens': 54, 'total_tokens': 510, 'latency_ms': 1464, 'response_id': 'chatcmpl-E99XgxTK2WH3K0BKmaf6TyByc8JXs'}


In [57]:
def load_completed_records(
    checkpoint_path: Path,
) -> list[dict[str, object]]:
    if not checkpoint_path.exists():
        return []

    records: list[dict[str, object]] = []

    with checkpoint_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            cleaned_line = line.strip()

            if cleaned_line:
                records.append(
                    json.loads(cleaned_line)
                )

    return records


def append_checkpoint(
    checkpoint_path: Path,
    record: dict[str, object],
) -> None:
    with checkpoint_path.open(
        "a",
        encoding="utf-8",
    ) as file:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

In [24]:
SAMPLE_SIZE = 220
TARGET_GROUND_TRUTH_SIZE = 180

In [58]:
completed_records = load_completed_records(
    CHECKPOINT_PATH
)

completed_chunk_ids = {
    str(record["chunk_id"])
    for record in completed_records
}

print(
    "Already completed:",
    len(completed_chunk_ids),
)

errors: list[dict[str, str]] = []

for _, row in tqdm(
    evaluation_sample.iterrows(),
    total=len(evaluation_sample),
):
    chunk_id = str(row["chunk_id"])

    if chunk_id in completed_chunk_ids:
        continue
    if len(completed_chunk_ids) >= TARGET_GROUND_TRUTH_SIZE:
     break
    try:
        generated, metadata = generate_question(
            row
        )

        if not generated.is_relevant:
            errors.append(
                {
                    "chunk_id": chunk_id,
                    "error": "Skipped: not relevant to PM Playbook",
                }
            )
            continue
        record = {
            "question_id": (
                f"question-{chunk_id}"
            ),
            "question": generated.question.strip(),
            "expected_answer": (
                generated.expected_answer.strip()
            ),
            "topic": generated.topic.strip(),
            "chunk_id": chunk_id,
            "episode_id": str(row["episode_id"]),
            "guest": str(row["guest"]),
            "episode_title": str(
                row["episode_title"]
            ),
            "speaker_name": str(
                row["speaker_name"]
            ),
            "word_count": int(
                row["word_count"]
            ),
            "is_relevant": generated.is_relevant,
            "length_bucket": str(
                row["length_bucket"]
            ),
            "source_text": str(row["text"]),
            **metadata,
        }

        append_checkpoint(
            CHECKPOINT_PATH,
            record,
        )

        completed_chunk_ids.add(
            chunk_id
        )

    except (
        OpenAIError,
        RuntimeError,
        ValueError,
    ) as error:
        errors.append(
            {
                "chunk_id": chunk_id,
                "error": (
                    f"{type(error).__name__}: {error}"
                ),
            }
        )

        print(
            f"\nFailed chunk {chunk_id}: {error}"
        )

print("Generation completed.")
print("Successful:", len(completed_chunk_ids))
print("Errors:", len(errors))

Already completed: 0


  0%|          | 0/260 [00:00<?, ?it/s]

Generation completed.
Successful: 180
Errors: 42


In [61]:
checkpoint_records = load_completed_records(
    CHECKPOINT_PATH
)

checkpoint_df = pd.DataFrame(
    checkpoint_records
)

print(
    checkpoint_df["is_relevant"]
    .value_counts(dropna=False)
)

print(
    "Accepted:",
    checkpoint_df["is_relevant"]
    .fillna(False)
    .sum(),
)

print(
    "Processed:",
    len(checkpoint_df),
)

is_relevant
True    180
Name: count, dtype: int64
Accepted: 180
Processed: 180


In [62]:
required_columns = {
    "question_id",
    "question",
    "expected_answer",
    "topic",
    "chunk_id",
    "episode_id",
    "guest",
    "episode_title",
    "speaker_name",
    "source_text",
    "is_relevant",
}

missing_columns = (
    required_columns
    - set(ground_truth.columns)
)

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

assert len(ground_truth) == 180
assert ground_truth["chunk_id"].is_unique
assert ground_truth["question_id"].is_unique
assert ground_truth["is_relevant"].all()
assert ground_truth["question"].str.strip().ne("").all()
assert (
    ground_truth["expected_answer"]
    .str.strip()
    .ne("")
    .all()
)

print("Records:", len(ground_truth))
print(
    "Unique questions:",
    ground_truth["question"].nunique(),
)
print(
    "Unique episodes:",
    ground_truth["episode_id"].nunique(),
)
print(
    "Length distribution:"
)
print(
    ground_truth["length_bucket"]
    .value_counts()
    .sort_index()
)
print(
    "Topic count:",
    ground_truth["topic"].nunique(),
)
print(
    "Total input tokens:",
    ground_truth["input_tokens"].sum(),
)
print(
    "Total output tokens:",
    ground_truth["output_tokens"].sum(),
)
print(
    "Average latency:",
    round(
        ground_truth["latency_ms"].mean()
    ),
    "ms",
)

AssertionError: Missing columns: {'is_relevant'}

In [64]:
ground_truth = checkpoint_df.copy()

# Keep only accepted records when the relevance field exists.
if "is_relevant" in ground_truth.columns:
    ground_truth = ground_truth[
        ground_truth["is_relevant"].fillna(False)
    ].copy()
else:
    # Older checkpoint format contained only accepted records.
    ground_truth["is_relevant"] = True

# Create question_id when it was not saved in the checkpoint.
if "question_id" not in ground_truth.columns:
    ground_truth["question_id"] = (
        "question-"
        + ground_truth["chunk_id"].astype(str)
    )

# Older code may have used text instead of source_text.
if (
    "source_text" not in ground_truth.columns
    and "text" in ground_truth.columns
):
    ground_truth["source_text"] = ground_truth["text"]

# Ensure optional generation fields exist.
for column, default_value in {
    "question": "",
    "expected_answer": "",
    "topic": "",
}.items():
    if column not in ground_truth.columns:
        ground_truth[column] = default_value

ground_truth = ground_truth.drop_duplicates(
    subset=["chunk_id"],
    keep="last",
)

ground_truth = ground_truth.head(
    TARGET_GROUND_TRUTH_SIZE
).reset_index(drop=True)

print("Ground-truth records:", len(ground_truth))
print("Columns:", sorted(ground_truth.columns.tolist()))

Ground-truth records: 180
Columns: ['chunk_id', 'episode_id', 'episode_title', 'expected_answer', 'guest', 'input_tokens', 'is_relevant', 'latency_ms', 'length_bucket', 'model', 'output_tokens', 'question', 'question_id', 'response_id', 'source_text', 'speaker_name', 'topic', 'total_tokens', 'word_count']


In [65]:
required_columns = {
    "question_id",
    "question",
    "expected_answer",
    "topic",
    "chunk_id",
    "episode_id",
    "guest",
    "episode_title",
    "speaker_name",
    "source_text",
    "is_relevant",
}

missing_columns = (
    required_columns
    - set(ground_truth.columns)
)

if missing_columns:
    raise ValueError(
        "Ground-truth data is missing required columns: "
        f"{sorted(missing_columns)}"
    )

assert len(ground_truth) == TARGET_GROUND_TRUTH_SIZE
assert ground_truth["chunk_id"].is_unique
assert ground_truth["question_id"].is_unique
assert ground_truth["is_relevant"].all()
assert ground_truth["question"].str.strip().ne("").all()
assert (
    ground_truth["expected_answer"]
    .str.strip()
    .ne("")
    .all()
)

print("Validation passed.")
print("Records:", len(ground_truth))
print("Unique questions:", ground_truth["question"].nunique())
print("Unique episodes:", ground_truth["episode_id"].nunique())

Validation passed.
Records: 180
Unique questions: 179
Unique episodes: 139


In [66]:
duplicate_questions = ground_truth[
    ground_truth["question"].duplicated(
        keep=False
    )
].sort_values("question")

print(
    duplicate_questions[
        [
            "question",
            "expected_answer",
            "guest",
            "episode_title",
            "chunk_id",
        ]
    ].to_string(index=False)
)

                                                        question                                                                                                                                                                             expected_answer     guest                                                                   episode_title                         chunk_id
What does Eric Ries suggest is worse than having a startup fail?                                  According to Eric Ries, being in a company that you hate and cannot leave, often described as a 'zombie company,' is far worse than having a startup fail. Eric Ries Reflections on a movement | Eric Ries (creator of the Lean Startup methodology) 703c5853430fb4f6563b0bb526f8d85e
What does Eric Ries suggest is worse than having a startup fail? According to Eric Ries, being in a company that you hate and can't leave, or being involved in a business that becomes something abhorrent to you, is far worse than having a startup f

In [67]:
chunk_id_to_remove = "703c5853430fb4f6563b0bb526f8d85e"

ground_truth = ground_truth[
    ~ground_truth["chunk_id"].eq(
        chunk_id_to_remove
    )
].reset_index(drop=True)

In [68]:
accepted_records = checkpoint_df[
    checkpoint_df["is_relevant"].fillna(False)
].copy()

replacement_candidates = accepted_records[
    ~accepted_records["chunk_id"].isin(
        ground_truth["chunk_id"]
    )
].copy()

print(
    replacement_candidates[
        [
            "question",
            "expected_answer",
            "guest",
            "episode_title",
            "chunk_id",
        ]
    ].head(10)
)

                                             question  \
62  What does Eric Ries suggest is worse than havi...   

                                      expected_answer      guest  \
62  According to Eric Ries, being in a company tha...  Eric Ries   

                                        episode_title  \
62  Reflections on a movement | Eric Ries (creator...   

                            chunk_id  
62  703c5853430fb4f6563b0bb526f8d85e  


In [69]:
existing_questions = set(
    ground_truth["question"]
    .str.strip()
    .str.casefold()
)

replacement_candidates = replacement_candidates[
    ~replacement_candidates["question"]
    .str.strip()
    .str.casefold()
    .isin(existing_questions)
].copy()

replacement = replacement_candidates.iloc[[0]]

ground_truth = pd.concat(
    [
        ground_truth,
        replacement,
    ],
    ignore_index=True,
)

IndexError: positional indexers are out-of-bounds

In [18]:
ground_truth_records = load_completed_records(
    CHECKPOINT_PATH
)

ground_truth = pd.DataFrame(
    ground_truth_records
)

ground_truth = ground_truth.drop_duplicates(
    subset=["chunk_id"],
    keep="last",
)

sample_order = {
    chunk_id: index
    for index, chunk_id in enumerate(
        evaluation_sample["chunk_id"]
        .astype(str)
        .tolist()
    )
}

ground_truth["_sample_order"] = (
    ground_truth["chunk_id"]
    .astype(str)
    .map(sample_order)
)

ground_truth = (
    ground_truth
    .sort_values("_sample_order")
    .drop(columns=["_sample_order"])
    .reset_index(drop=True)
)

records = ground_truth.to_dict(
    orient="records"
)

with GROUND_TRUTH_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        records,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Ground-truth records:", len(records))
print("Saved to:", GROUND_TRUTH_PATH)

Ground-truth records: 180
Saved to: ../data/evaluation/ground-truth.json


In [19]:
required_columns = {
    "question_id",
    "question",
    "expected_answer",
    "topic",
    "chunk_id",
    "episode_id",
    "guest",
    "episode_title",
    "speaker_name",
    "source_text",
}

missing_columns = (
    required_columns
    - set(ground_truth.columns)
)

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

assert ground_truth["chunk_id"].is_unique
assert ground_truth["question_id"].is_unique
assert ground_truth["question"].str.strip().ne("").all()
assert (
    ground_truth["expected_answer"]
    .str.strip()
    .ne("")
    .all()
)

print("Records:", len(ground_truth))
print(
    "Unique episodes:",
    ground_truth["episode_id"].nunique(),
)
print(
    "Unique questions:",
    ground_truth["question"].nunique(),
)
print(
    "Total input tokens:",
    ground_truth["input_tokens"].sum(),
)
print(
    "Total output tokens:",
    ground_truth["output_tokens"].sum(),
)
print(
    "Average latency:",
    round(
        ground_truth["latency_ms"].mean(),
    ),
    "ms",
)

ground_truth[
    [
        "question",
        "expected_answer",
        "topic",
        "guest",
        "length_bucket",
    ]
].head(10)

Records: 180
Unique episodes: 135
Unique questions: 180
Total input tokens: 97791
Total output tokens: 11794
Average latency: 1467 ms


,question,expected_answer,topic,guest,length_bucket
0,What realization did Deb Liu have about her jo...,Deb Liu realized she was bored of her job and ...,career transition,Deb Liu,short
1,What does John Cutler say about the perception...,John Cutler suggests that individuals should n...,workplace culture,John Cutler,short
2,What does Sarah Tavel define as the 'core acti...,The core action of a product is the fundamenta...,user engagement,Sarah Tavel,long
3,What does Grant Lee suggest as an important re...,Grant Lee suggests the book '7 Powers' by Hami...,strategy,Grant Lee,medium
4,What key realization is necessary for personal...,The key realization necessary for personal tra...,personal transformation,Andy Johns,long
5,What are the two major groups involved in grow...,"The two major groups are growth R&D, which inc...",growth strategy,Archie Abrams,short
6,How does Guillermo Rauch suggest leveraging AI...,Guillermo Rauch suggests that by drawing inspi...,design,Guillermo Rauch,short
7,What technique does Andy Johns suggest for ach...,He suggests using pen and paper to write to yo...,self-reflection,Andy Johns,medium
8,How did Eventbrite improve its performance mar...,Eventbrite unified their separate direct respo...,marketing strategy,Casey Winters,medium
9,What insight does David Placek provide about t...,David Placek explains that the Sonos name was ...,branding,David Placek,medium


In [20]:
review_sample = ground_truth.sample(
    n=min(20, len(ground_truth)),
    random_state=42,
)

for row in review_sample.itertuples(index=False):
    print("=" * 100)
    print("QUESTION:")
    print(row.question)

    print("\nEXPECTED ANSWER:")
    print(row.expected_answer)

    print("\nSOURCE:")
    print(row.guest, "|", row.episode_title)
    print(row.source_text[:700])
    print()

QUESTION:
What role did Gainsight play in addressing customer success within organizations?

EXPECTED ANSWER:
Gainsight developed a customer success platform to solve the specific problem of how customer success teams can make their customers successful.

SOURCE:
Barbra Gago | Category creation and brand building | Barbra Gago (Pando, Miro, Greenhouse, Culture Amp)
In the early days, I was working on market automation, for example, so Marketo, now HubSpot, fits into that, even though back then, they were more of this inbound marketing platform/category performance management, which is the area of Pando. We've talked a lot about people, products. I think Gainsight did a great job with the customer success platform, so this is a different problem that they were solving within the organization for a group. The customer success team, basically how are they going to make their customers successful?

QUESTION:
How does the definition of product management relate to understanding what success